# Supervised Attention Alignment Training

This notebook demonstrates supervised training of the ALBERT model with attention alignment on atom-mapped reaction SMILES.

## Setup

1. **Install dependencies**: Run cell 1 to install `agave_chem`, `torch`, and `pandas`.
2. **Configure paths**: Update `PRETRAINED_MODEL_PATH`, `TRAINING_DATA_FILE`, and `SAVE_DIR` in cell 2.
3. **Set target layer**: Update `TARGET_LAYER` to the layer identified by `layer_head_identification.ipynb`.
4. **Tune hyperparameters**: Adjust `NUM_EPOCHS`, `BATCH_SIZE`, etc. in cell 2.
5. **Resume from checkpoint**: Set `RESUME_FROM_CHECKPOINT` to a `.pt` checkpoint path.
6. **Early stopping**: Set `EARLY_STOPPING_PATIENCE` > 0 to stop when validation loss plateaus.

## Workflow

1. Run `unsupervised_training.ipynb` first to pre-train the MLM model.
2. Run `layer_head_identification.ipynb` to find the best attention layer.
3. Run this notebook with the identified `TARGET_LAYER` to train the supervised model.

In [ ]:
!pip install -e ../.. --force-reinstall --no-cache-dir
!python -m pip install -U --no-cache-dir torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -U typing_extensions pydantic pydantic-core pandas

In [ ]:
import os
import sys
from pathlib import Path

WORKFLOWS_DIR = Path.cwd().parent
REPO_ROOT = Path.cwd().parent.parent
for _path in (WORKFLOWS_DIR, REPO_ROOT):
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))

from model_training_scripts.albert_mapper_supervised_training import (
    main_supervised,
)
from model_training_scripts.albert_mapper_unuspervised_training import TrainingConfig
from model_training_scripts.cli_utils import read_lines, split_data

from agave_chem.mappers.neural.constants import smiles_token_to_id_dict
from agave_chem.mappers.neural.model import SupervisedConfig
from agave_chem.mappers.neural.tokenizer import CustomTokenizer

TARGET_LAYER = 11

NUM_EPOCHS = 30
BATCH_SIZE = 64
WARMUP_STEPS = 10000
LOGGING_STEPS = 100
TRAIN_PCT = 0.99
MAX_LENGTH = 384
NUM_WORKERS = 8
PREFETCH_FACTOR = 4
MASKING_MODE = "span"
SEED = 42
SAVE_BEST_MODEL = True
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 0.001
RESUME_FROM_CHECKPOINT = None
PRETRAINED_MODEL_PATH = "/workspace/saved_models/albert-04-01-2026/checkpoint-epoch-3"
TRAINING_DATA_FILE = "/workspace/data/uspto_all_reactions_training_filtered.txt"
SAVE_DIR = "/workspace/saved_models/albert-supervised-04-01-2026"
os.makedirs(SAVE_DIR, exist_ok=True)

tokenizer = CustomTokenizer(smiles_token_to_id_dict)

In [ ]:
rxns = read_lines(TRAINING_DATA_FILE)

rxns_train, rxns_val = split_data(
    rxns=rxns,
    train_pct=TRAIN_PCT,
    shuffle=True,
    seed=SEED,
)
rxns_train_filtered = rxns_train
rxns_val_filtered = rxns_val

In [ ]:
training_config = TrainingConfig(
    output_dir=SAVE_DIR,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    warmup_steps=WARMUP_STEPS,
    logging_steps=LOGGING_STEPS,
    seed=SEED,
    save_best_model=SAVE_BEST_MODEL,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
)

supervised_config = SupervisedConfig()
supervised_config.target_layer = TARGET_LAYER

main_supervised(
    train_texts=rxns_train_filtered,
    val_texts=rxns_val_filtered,
    training_config=training_config,
    supervised_config=supervised_config,
    pretrained_model_path=PRETRAINED_MODEL_PATH,
    max_length=MAX_LENGTH,
    num_workers=NUM_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
    masking_mode=MASKING_MODE,
    resume_from_checkpoint=RESUME_FROM_CHECKPOINT,
)